## 2. Analyse comparative (avant / après)

Cette partie compare la situation actuelle (baseline du milestone 2) au planning consolidé produit par l'algorithme du milestone 3. 
Les deux sont chargés depuis leurs fichiers. 
Les mêmes indicateurs sont calculés des deux côtés, avec la même fonction (`preparer`) : coût avec la grille tarifaire du milestone 2, taux de remplissage, semaine et jour.

Les **134 camions réservés** (lignes sans poids) sont identiques avant et après. Ils sont **exclus des distributions** de remplissage, car on ne connaît pas leur chargement, mais **inclus dans les coûts totaux**.

Dans un premier temps, nous récupérons tous les fichiers nécessaires : la baseline et notre planning optimisé.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

#  Fichiers 
BASELINE_PATH = "../data/baseline/baseline_camion.csv"     # avant : une ligne par camion (milestone 2)
SORTIE2_PATH = "../data/gold/sortie2_camions.csv"           # après : une ligne par camion × material × quai (milestone 3)
SILVER_CLEAN_PATH = "../data/silver/released_schedule_cleaned.csv"
QUARANTAINE_PATH = "../data/silver/released_schedule_quarantine.csv"
RATE_SHEET_PATH = "../data/bronze/rate_sheet.csv"
DISTANCES_PATH = "../data/bronze/distances.csv"

# Paramètres (mêmes valeurs qu'aux milestones 2 et 3)
POIDS_MAX = 24500     # kg
VOLUME_MAX = 110.4    # m³
FREIGHT_VALUE_MARGIN = 2.3923   # +239,23 %
DIESEL_SURCHARGE = 1.1943

COULEURS = {"Avant": "#b8c4cc", "Après": "#3f7489"}   # gris = baseline, bleu = planning consolidé

#  Grille tarifaire 
couts = pd.read_csv(RATE_SHEET_PATH, index_col="km_max")
TRANCHES_KM = [int(km) for km in couts.index if not km.startswith(">")]
TRANCHES_KG = [int(kg) for kg in couts.columns if not kg.startswith(">")]
POIDS_PAR_DEFAUT = TRANCHES_KG[-1] + 1      # poids inconnu -> tranche > 4 000 kg
DISTANCE_PAR_DEFAUT = TRANCHES_KM[-1] + 1   # distance inconnue -> tranche > 650 km


def tranche(valeur, tranches):
    # Première tranche >= valeur (on majore), sinon la tranche '>max'
    for borne in tranches:
        if valeur <= borne:
            return str(borne)
    return f">{tranches[-1]}"


def cout_transport(poids, distance):
    # Prix d'un camion en € = coût de la grille × (1 + marge) × surcharge diesel
    cout_base = couts.loc[tranche(distance, TRANCHES_KM), tranche(poids, TRANCHES_KG)]
    return float(round(cout_base * (1 + FREIGHT_VALUE_MARGIN) * DIESEL_SURCHARGE, 2))


# Mise en forme 
def nombre(x, decimales=1):
    return f"{x:,.{decimales}f}".replace(",", " ").replace(".", ",").removesuffix(",0")
def pourcent(x, decimales=1):
    return f"{x:.{decimales}%}".replace(".", ",").replace("%", " %")
def eur(x):
    return f"{x:,.0f} €".replace(",", " ")


#  Avant : baseline du milestone 2 
distances = pd.read_csv(DISTANCES_PATH, encoding="utf-8-sig").set_index("supplier_id")["distance"]
avant = pd.read_csv(BASELINE_PATH, parse_dates=["date"])[["camion_id", "supplier_id", "plant", "date", "poids", "volume", "reserve"]]

#  Après : planning consolidé du milestone 3, reconstruit camion par camion depuis la sortie 2 
sortie2 = pd.read_csv(SORTIE2_PATH, parse_dates=["date_livraison"])
sortie2 = sortie2.rename(columns={"Material": "material", "Quai": "quai", "Besoins regroupés": "besoins",
                                  "Quantité": "quantite", "Poids (kg)": "poids_kg"})   # noms des colonnes du livrable

# La sortie 2 n'a pas le volume : on le reprend dans les lignes de besoin (silver + quarantaine).
# Une ligne de la sortie 2 peut regrouper plusieurs dates de besoin ("2026-11-16, 2026-11-17") : on la découpe par date.
besoins = pd.concat([pd.read_csv(SILVER_CLEAN_PATH, parse_dates=["need_date"]),
                     pd.read_csv(QUARANTAINE_PATH, parse_dates=["need_date"])], ignore_index=True)
CLE = ["need_date", "supplier_id", "plant", "material", "unloading_point"]
volumes = besoins.groupby(CLE)["volume"].sum(min_count=1)

detail = (sortie2.rename(columns={"fournisseur": "supplier_id", "usine": "plant", "quai": "unloading_point"})
                 .assign(need_date=lambda d: d["besoins"].str.split(", "))
                 .explode("need_date"))
detail["need_date"] = pd.to_datetime(detail["need_date"])
detail["volume"] = volumes.reindex(pd.MultiIndex.from_frame(detail[CLE])).values

apres = (detail.groupby("camion_id")
               .agg(supplier_id=("supplier_id", "first"), plant=("plant", "first"), date=("date_livraison", "first"),
                    volume=("volume", lambda v: v.sum(min_count=1)), reserve=("reserve", "first"))
               .reset_index())
apres["poids"] = sortie2.groupby("camion_id")["poids_kg"].sum(min_count=1).loc[apres["camion_id"]].values  # NaN = réservé


#  Indicateurs communs avant/après
def preparer(camions, version):
    c = camions.copy()
    c["version"] = version
    c["distance"] = c["supplier_id"].map(distances).fillna(DISTANCE_PAR_DEFAUT)   # sans distance -> > 650 km
    c["cout"] = [cout_transport(POIDS_PAR_DEFAUT if r else p, d)                   # réservé -> > 4 000 kg
                 for p, d, r in zip(c["poids"], c["distance"], c["reserve"])]
    c["remplissage_poids"] = c["poids"] / POIDS_MAX
    c["remplissage_volume"] = c["volume"] / VOLUME_MAX
    c["semaine"] = c["date"].dt.isocalendar().week
    c["jour"] = c["date"].dt.day_of_week          # 0 = lundi
    return c


camions = pd.concat([preparer(avant, "Avant"), preparer(apres, "Après")], ignore_index=True)
connus = camions[~camions["reserve"]]   # camions dont le poids est connu, les seuls pour analyser le remplissage

#  Contrôle pour etre sur qu'on retrouve bien les chiffres des milestones 2 et 3 
controle = camions.groupby("version", sort=False).agg(camions=("camion_id", "size"), dont_reserves=("reserve", "sum"),
                                                      poids_total_t=("poids", "sum"), cout=("cout", "sum"))
controle["poids_total_t"] = (controle["poids_total_t"] / 1000).round(0)   # tonnes : l'écart d'arrondi (< 1 kg) disparaît
controle

Les 134 camions réservés sont exclus : on ne connaît pas leur poids. Il reste les camions dont le poids est connu, 1 607 avant (1 741 − 134) et 653 après (787 − 134). Comme leur nombre est très différent, on affiche la **part** des camions de chaque version dans chaque classe, et non leur nombre.

### Nouvelles distributions (taux de remplissage, coûts, ...)

#### Poids et taux de remplissage des camions

Le taux de remplissage d'un camion, c'est son poids divisé par 24,5 t. Les classes suivent les tranches de poids de la grille tarifaire, car le milestone 2 a montré que le seuil de 4 000 kg est décisif : au-delà, le prix d'un camion ne bouge presque plus. Sous chaque classe, on affiche le taux de remplissage correspondant.


In [ ]:
#  Poids et taux de remplissage par camion, selon les tranches de la grille tarifaire 
# Les bornes sont toutes des tranches de poids de la grille : dans une classe, le prix d'un camion change peu
BORNES_KG = [0, 100, 300, 500, 1000, 2000, 4000, POIDS_MAX]
# Chaque classe affiche aussi son taux de remplissage (poids / 24,5 t)
LIBELLES_KG = [f"{a}-{b} kg\n({pourcent(BORNES_KG[i] / POIDS_MAX)}-{pourcent(BORNES_KG[i + 1] / POIDS_MAX)})"
               for i, (a, b) in enumerate(zip(["0", "100", "300", "500", "1 000", "2 000", "4 000"],
                                              ["100", "300", "500", "1 000", "2 000", "4 000", "24 500"]))]

effectifs = connus.groupby("version")["camion_id"].size()
classes_kg = (pd.crosstab(pd.cut(connus["poids"], bins=BORNES_KG, labels=LIBELLES_KG, include_lowest=True),
                          connus["version"], normalize="columns")     # part des camions de chaque version
              [["Avant", "Après"]])
mediane_kg = connus.groupby("version")["poids"].median()

ax = classes_kg.mul(100).plot(kind="bar", color=[COULEURS[v] for v in classes_kg.columns], edgecolor="black",
                              figsize=(11, 5.2), rot=0, width=0.8)
for conteneur in ax.containers:
    ax.bar_label(conteneur, fmt="%.0f %%", fontsize=8, padding=2)
ax.axvline(len(LIBELLES_KG) - 1.5, color="red", linestyle="--")      # entre « 2 000-4 000 » et « > 4 000 »
ax.annotate("fin de la grille tarifaire :\nau-delà, prix quasi fixe", xy=(len(LIBELLES_KG) - 1.45, 22),
            color="red", fontsize=9)
ax.set_title(f"Poids et taux de remplissage des camions, selon les tranches de la grille tarifaire\n"
             f"médiane : {nombre(mediane_kg['Avant'], 0)} kg ({pourcent(mediane_kg['Avant'] / POIDS_MAX)}) avant → "
             f"{nombre(mediane_kg['Après'], 0)} kg ({pourcent(mediane_kg['Après'] / POIDS_MAX)}) après "
             f"(hors camions réservés)")
ax.set_xlabel("Poids du camion (taux de remplissage = poids / 24,5 t)")
ax.set_ylabel("Part des camions (%)")
ax.legend(ax.containers, [f"Avant : baseline ({effectifs['Avant']} camions)",
                          f"Après : planning consolidé ({effectifs['Après']} camions)"], loc="upper center")
ax.grid(axis="y", alpha=0.5)
ax.set_axisbelow(True)
ax.margins(y=0.12)
plt.tight_layout()
plt.show()

# ---------- Chiffres pour l'analyse ----------
for version in ["Avant", "Après"]:
    p = connus.loc[connus["version"] == version, "poids"]
    print(f"{version} : médiane {nombre(p.median(), 0)} kg ({pourcent(p.median() / POIDS_MAX)}), "
          f"moyenne {nombre(p.mean(), 0)} kg | ≤ 500 kg : {pourcent((p <= 500).mean())} | "
          f"> 4 000 kg : {(p > 4000).sum()} camions ({pourcent((p > 4000).mean())}), "
          f"soit {pourcent(p[p > 4000].sum() / p.sum())} du poids transporté")

# Pourquoi le remplissage reste bas : le poids qu'un couple a à livrer en une semaine
poids_semaine = connus[connus["version"] == "Après"].groupby(["supplier_id", "plant", "semaine"])["poids"].sum()
print(f"Poids médian à livrer par couple et par semaine : {nombre(poids_semaine.median(), 0)} kg ; "
      f"{pourcent((poids_semaine <= 4000).mean())} des couples-semaines sous 4 000 kg")

#### Analyse : poids et taux de remplissage

**La distribution se décale vers la droite.** Le poids médian d'un camion passe de **323 kg à 687 kg**, soit un taux de remplissage médian de **1,3 % à 2,8 %** (×2,1). Les petits envois reculent nettement : 64 % des camions pesaient moins de 500 kg (2 % de remplissage) dans la baseline, contre 42 % après consolidation. La classe 100-300 kg, la plus fréquente avant (30 % des camions), tombe à 19 %.

**Le même poids part dans 2,5 fois moins de camions.** Les 1 588 t à livrer sont les mêmes avant et après (aucune ligne perdue, voir la partie 1.4). Elles partent dans 653 camions au lieu de 1 607. Le poids moyen d'un camion passe donc de 988 kg à 2 433 kg.

**La part des camions au-delà de 4 000 kg est multipliée par 3,5 : de 4,4 % à 15,2 %** (de 71 à 99 camions). C'est le chiffre qui compte pour le prix : au-delà de 4 000 kg, le prix d'un camion est quasi fixe. Ces camions transportent maintenant **69 % du poids total**, contre 47 % avant : la majorité des marchandises voyage dans la tranche de prix la plus avantageuse au kilo.

**Le remplissage reste pourtant faible**, car les couples ont peu à livrer : en médiane, un couple n'a que 687 kg à livrer par semaine, et 86 % des couples-semaines restent sous 4 000 kg. Même regroupés dans un seul camion par semaine, ces besoins ne peuvent pas remplir un camion de 24,5 t. Pour aller plus loin, il faudrait lever deux contraintes : mélanger plusieurs fournisseurs dans un même camion (C4) ou livrer plus de 7 jours en avance (C6).

#### Coût de transport par tranche de poids

Le graphe précédent compte des camions ; celui-ci additionne leur coût, avec les mêmes tranches de poids. Sous chaque tranche, on affiche le coût moyen à la tonne transportée : c'est lui qui montre l'effet de la grille tarifaire dégressive. Les camions réservés sont exclus (leur coût est identique avant et après : 33 506 €).

In [ ]:
#  Où part l'argent : coût total par tranche de poids, avant / après 
classe_cout = pd.cut(connus["poids"], bins=BORNES_KG, labels=LIBELLES_KG, include_lowest=True)
cout_classe = connus.groupby([classe_cout, "version"], observed=False)["cout"].sum().unstack()[["Avant", "Après"]]
cout_tonne = connus.groupby("version").apply(lambda c: c["cout"].sum() / c["poids"].sum() * 1000)   # € par tonne
cout_tonne_classe = connus.groupby(classe_cout, observed=False).apply(lambda c: c["cout"].sum() / c["poids"].sum() * 1000)

ax = cout_classe.plot(kind="bar", color=[COULEURS[v] for v in cout_classe.columns], edgecolor="black",
                      figsize=(11, 5.2), rot=0, width=0.8)
for conteneur in ax.containers:
    ax.bar_label(conteneur, labels=[eur(v) for v in conteneur.datavalues], fontsize=8, padding=2)
ax.axvline(len(LIBELLES_KG) - 1.5, color="red", linestyle="--")
ax.set_xticks(range(len(LIBELLES_KG)),
              [f"{l.split(chr(10))[0]}\n{nombre(t, 0)} €/t" for l, t in zip(LIBELLES_KG, cout_tonne_classe)])
total = connus.groupby("version")["cout"].sum()
ax.set_title(f"Coût de transport par tranche de poids des camions (hors camions réservés)\n"
             f"total : {eur(total['Avant'])} avant → {eur(total['Après'])} après ; "
             f"coût à la tonne : {nombre(cout_tonne['Avant'], 0)} €/t → {nombre(cout_tonne['Après'], 0)} €/t")
ax.set_xlabel("Poids du camion (sous chaque tranche : coût moyen à la tonne transportée)")
ax.set_ylabel("Coût total sur la période (€)")
ax.legend(ax.containers, ["Avant : baseline", "Après : planning consolidé"], loc="upper center", ncol=2)
ax.grid(axis="y", alpha=0.5)
ax.set_axisbelow(True)
ax.set_ylim(0, cout_classe.values.max() * 1.25)
ax.yaxis.set_major_formatter(lambda v, _: nombre(v, 0))
plt.tight_layout()
plt.show()

# Chiffres pour l'analyse
for version in ["Avant", "Après"]:
    c = connus[connus["version"] == version]
    petits = c["poids"] <= 500
    print(f"{version} : coût total {eur(c['cout'].sum())}, coût médian d'un camion {nombre(c['cout'].median(), 0)} €, "
          f"coût à la tonne {nombre(cout_tonne[version], 0)} €/t | camions ≤ 500 kg : {pourcent(petits.mean())} des camions, "
          f"{pourcent(c.loc[petits, 'poids'].sum() / c['poids'].sum())} du poids, "
          f"{pourcent(c.loc[petits, 'cout'].sum() / c['cout'].sum())} du coût ({eur(c.loc[petits, 'cout'].sum())})")

#### Analyse : coût par tranche de poids

**La grille tarifaire rend les petits camions très chers à la tonne.** Une tonne transportée dans un camion de moins de 100 kg coûte en moyenne 778 €, contre 23 € dans un camion de plus de 4 000 kg : environ 34 fois moins. Ce coût à la tonne dépend de la grille, pas du moteur. L'économie vient donc du fait que le poids est déplacé vers les tranches les moins chères.

**C'est ce que fait la consolidation.** Dans la baseline, les camions de moins de 500 kg représentaient 64 % des camions et seulement 13 % du poids transporté, mais **39 % du coût** (55 807 €). Après consolidation, ils ne pèsent plus que 20 % du coût (16 510 €). Toutes les tranches sous 4 000 kg baissent, surtout les plus légères.

**Seule la tranche au-delà de 4 000 kg coûte plus cher après** (17 182 € → 25 064 €). C'est voulu : elle compte plus de camions, mais c'est la tranche où la tonne coûte le moins cher. Elle transporte désormais 69 % du poids pour 30 % du coût.

**Au total, hors camions réservés, le coût passe de 143 815 € à 82 612 € (−43 %), et le coût à la tonne de 91 €/t à 52 €/t.** Chaque camion coûte un peu plus cher (médiane de 71 € à 101 €, car il est plus lourd), mais il y en a 2,5 fois moins.

#### Camions reçus par jour et par usine

Ce graphe reprend la charge des usines du milestone 2, jour par jour et usine par usine. Les camions réservés sont comptés : l'usine les reçoit à ses quais comme les autres. Le milestone 3 comparait le planning avant et après lissage. Ici, on le compare à la situation actuelle (baseline).

In [ ]:
#  Camions reçus chaque jour ouvré, par usine : avant / après 
# Tous les camions sont comptés (camions réservés compris) 
par_jour = (camions.groupby(["plant", "date", "version"]).size().unstack(fill_value=0)
                   .reindex(columns=["Avant", "Après"]))
dates = sorted(camions["date"].unique())
lundis = [d for d in dates if pd.Timestamp(d).day_of_week == 0]

fig, axes = plt.subplots(2, 2, figsize=(14, 7.5), sharex=True)
for ax, plant in zip(axes.flat, sorted(camions["plant"].unique())):
    serie = par_jour.loc[plant].reindex(dates, fill_value=0)
    x = np.arange(len(dates))
    ax.bar(x, serie["Avant"], color=COULEURS["Avant"], edgecolor="black", linewidth=0.4, label="Avant : baseline")
    ax.bar(x, serie["Après"], width=0.45, color=COULEURS["Après"], label="Après : planning consolidé")
    for i, d in enumerate(dates):                         # un trait au début de chaque semaine
        if pd.Timestamp(d).day_of_week == 0:
            ax.axvline(i - 0.5, color="grey", linewidth=0.6, linestyle=":")
    ax.set_title(f"{plant} : jour le plus chargé {serie['Avant'].max()} → {serie['Après'].max()} camions")
    ax.set_xticks([dates.index(l) for l in lundis], [pd.Timestamp(l).strftime("%d/%m") for l in lundis])
    ax.grid(axis="y", alpha=0.5)
    ax.set_axisbelow(True)
for ax in axes[:, 0]:
    ax.set_ylabel("Camions reçus")
for ax in axes[1]:
    ax.set_xlabel("Jours ouvrés (étiquette = lundi de chaque semaine)")
axes[0, 0].legend(loc="upper left")
fig.suptitle("Camions reçus par jour ouvré et par usine (camions réservés compris)", fontsize=14)
plt.tight_layout()
plt.show()

#  Chiffres pour l'analyse 
stats = par_jour.reset_index().melt(id_vars=["plant", "date"], var_name="version", value_name="nb")
stats = stats[stats["nb"] > 0]   # moyenne sur les jours où l'usine reçoit des camions (pas sur les jours hors planning)
print(stats.groupby(["plant", "version"])["nb"].agg(["max", "mean", "std"]).round(1).unstack())
total_jour = camions.groupby(["date", "version"]).size().unstack()[["Avant", "Après"]]
print(f"Toutes usines : jour le plus chargé {int(total_jour['Avant'].max())} ({total_jour['Avant'].idxmax():%d/%m}) → "
      f"{int(total_jour['Après'].max())} ({total_jour['Après'].idxmax():%d/%m})")

#### Analyse : camions reçus par jour et par usine

**Toutes les usines reçoivent beaucoup moins de camions chaque jour.** En moyenne, sur les jours où elles sont livrées, PLT_001 passe de 18,6 à 7,0 camions par jour, PLT_002 de 19,7 à 9,1, PLT_003 de 6,9 à 3,7 et PLT_004 de 18,5 à 6,3.

**Les pics disparaissent.** Le cas le plus net est PLT_002 : ses pics des 26 et 27/11 (53 et 54 camions, repérés dès le milestone 1) tombent à 15 et 13 camions, et l'écart-type de sa charge quotidienne est divisé par 4 (de 14,3 à 3,6). Le jour le plus chargé de chaque usine baisse fortement : de 31 à 11 camions pour PLT_001, de 54 à 15 pour PLT_002, de 16 à 6 pour PLT_003 et de 24 à 10 pour PLT_004. Toutes usines confondues, le jour le plus chargé passe de 118 camions (le 26/11) à 35 (le 23/11).

**Certains jours reçoivent plus de camions qu'avant**, par exemple PLT_002 le 18/11 (13 camions au lieu de 3). C'est l'effet du lissage : des jours fixes sont placés sur les jours creux pour équilibrer la semaine.

**Des livraisons commencent avant le planning, du 03 au 06/11.** Les besoins du début du planning sont livrés en avance, sur les jours fixes de la semaine précédente (57 camions, au plus 23 le même jour toutes usines confondues). Le premier jour du planning n'est donc plus un pic : le 09/11, les usines reçoivent 30 camions au total, contre 57 dans la baseline.

#### Jours de livraison des couples les plus actifs

Même carte de chaleur qu'au milestone 2 : les 10 couples qui livrent le plus souvent dans la baseline, avec à droite les mêmes couples après consolidation. Une livraison correspond à un couple et un jour : plusieurs camions le même jour comptent pour une seule livraison. Chaque case donne le nombre de livraisons ce jour de la semaine sur toute la période (le planning final peut en compter une de plus, car il livre aussi la semaine qui précède le planning).
Ces couples étaient ceux au plus fort potentiel : cette comparaison montre si nous avons su les optimiser.

In [ ]:
#  Jours de livraison des 10 couples les plus actifs 
JOURS = ["Lundi", "Mardi", "Mercredi", "Jeudi", "Vendredi"]
camions["couple"] = camions["supplier_id"] + " × " + camions["plant"]

# Une livraison = un couple, un jour (plusieurs camions le même jour comptent pour une livraison)
livraisons = camions.drop_duplicates(["version", "couple", "date"])
tables = {v: pd.crosstab(livraisons.loc[livraisons["version"] == v, "couple"],
                         livraisons.loc[livraisons["version"] == v, "jour"]).reindex(columns=range(5), fill_value=0)
          for v in ["Avant", "Après"]}
top_10 = tables["Avant"].sum(axis=1).nlargest(10).index      # mêmes couples qu'au milestone 2
nb_semaines = besoins["need_date"].dt.isocalendar().week.nunique()   # 6 semaines de besoins dans le planning

fig, axes = plt.subplots(1, 2, figsize=(13, 6.5), sharey=True)
for ax, version in zip(axes, ["Avant", "Après"]):
    heat = tables[version].reindex(top_10, fill_value=0)
    im = ax.imshow(heat.values, cmap="YlGnBu", vmin=0, vmax=nb_semaines, aspect="auto")
    for i in range(heat.shape[0]):
        for j in range(heat.shape[1]):
            v = heat.values[i, j]
            ax.text(j, i, v if v else "", ha="center", va="center", fontsize=9,
                    color="white" if v > nb_semaines / 2 else "black")
    ax.set_xticks(range(5), JOURS)
    ax.set_yticks(range(len(top_10)), top_10)
    titre = "Avant : baseline" if version == "Avant" else "Après : planning consolidé"
    ax.set_title(f"{titre}\n{int(heat.values.sum())} livraisons, soit {nombre(heat.values.sum() / nb_semaines / 10)} par couple et par semaine")
fig.colorbar(im, ax=axes, label=f"Nombre de livraisons sur les {nb_semaines} semaines", shrink=0.8)
fig.suptitle("Jours de livraison des 10 couples les plus actifs de la baseline", fontsize=14)
plt.show()

#  Chiffres pour l'analyse
par_sem = livraisons.groupby(["version", "couple", "semaine"])["date"].nunique()   # jours de livraison par semaine
moy = par_sem.groupby(["version", "couple"]).mean()
for v in ["Avant", "Après"]:
    print(f"{v} : {moy[v].mean():.2f} jours de livraison par semaine en moyenne (couples livrés), "
          f"{(moy[v] > 2).sum()} couples à plus de 2 jours par semaine sur {len(moy[v])}")

# Les couples qui livrent encore plus de 2 jours par semaine après : camions optimisés / camions réservés
apres_c = camions[camions["version"] == "Après"]
for couple in moy["Après"][moy["Après"] > 2].index:
    nb = apres_c[apres_c["couple"] == couple].groupby("reserve").size()
    print(f"  {couple} : {nb.get(False, 0)} camions optimisés, {nb.get(True, 0)} camions réservés")

#### Analyse : jours de livraison des couples les plus actifs

**Avant, ces couples livraient tous les jours.** Les 10 couples les plus actifs de la baseline livrent les 5 jours de la semaine, presque chaque semaine : 291 livraisons sur les 6 semaines, soit 4,8 par couple et par semaine. 8 d'entre eux livrent PLT_004.

**Après, chaque couple livre une seule fois par semaine, toujours le même jour.** Ces mêmes couples ne font plus que 61 livraisons. Chaque ligne n'a plus qu'une case foncée, le jour fixe du couple : par exemple le lundi pour SUP_00085 → PLT_004, ou le vendredi pour SUP_00011 → PLT_004. SUP_00107 → PLT_001 compte 7 mercredis : il livre aussi le mercredi 04/11, la semaine qui précède le planning, pour les premiers besoins.

**Un seul jour fixe suffit.** Les 10 couples ont tous un profil d'un seul jour, comme 172 couples sur 173 : la fenêtre de 7 jours permet à un camion par semaine de couvrir tous les besoins.

**Sur l'ensemble des couples**, le nombre moyen de jours de livraison par semaine passe de 2,0 à 1,1. Seuls 4 couples sur 180 livrent encore plus de 2 jours par semaine, contre 52 avant. Ces 4 couples ont surtout des lignes sans poids (par exemple 24 camions réservés sur 30 pour SUP_00129 → PLT_003) : leurs camions réservés partent le jour du besoin, en dehors des jours fixes. Pour eux, seule une meilleure qualité des données permettrait de regrouper les livraisons.

--- 

### Étude des économies réalisées (réalisé vs potentiel)

#### Les 10 couples les plus chers

Même classement qu'au milestone 2 : les 10 couples fournisseur × usine qui coûtent le plus cher dans la baseline, camions réservés compris. Pour chacun, on compare le coût avant et après consolidation. La partie hachurée est le coût des camions réservés (lignes sans poids) : il est identique avant et après, car ces lignes ne peuvent pas être consolidées.

In [ ]:
#  Coût des 10 couples les plus chers de la baseline : avant / après 
from matplotlib.patches import Patch

# Coût total de chaque couple, camions réservés compris (comme au milestone 2), et la part des camions réservés
cout_couple = camions.pivot_table(index="couple", columns="version", values="cout", aggfunc="sum", fill_value=0)[["Avant", "Après"]]
cout_reserve = (camions[camions["reserve"]]
                .pivot_table(index="couple", columns="version", values="cout", aggfunc="sum", fill_value=0)
                .reindex(index=cout_couple.index, columns=["Avant", "Après"], fill_value=0))
top_cher = cout_couple.nlargest(10, "Avant").index[::-1]          # du moins cher au plus cher (le plus cher en haut)

fig, ax = plt.subplots(figsize=(11, 6.5))
y = np.arange(len(top_cher))
h = 0.4
for decalage, version in [(h / 2, "Avant"), (-h / 2, "Après")]:
    total = cout_couple.loc[top_cher, version]
    reserve = cout_reserve.loc[top_cher, version]
    ax.barh(y + decalage, total - reserve, height=h, color=COULEURS[version], edgecolor="black",
            label="Avant : baseline" if version == "Avant" else "Après : planning consolidé")
    ax.barh(y + decalage, reserve, left=total - reserve, height=h, color=COULEURS[version], edgecolor="black",
            hatch="///", alpha=0.6)
    for yi, v in zip(y + decalage, total):
        ax.text(v, yi, f" {eur(v)}", va="center", fontsize=8)
ax.set_yticks(y, top_cher)
ax.set_xlabel("Coût total sur les 6 semaines (€)")
ax.xaxis.set_major_formatter(lambda v, _: nombre(v, 0))
avant_top, apres_top = cout_couple.loc[top_cher, "Avant"].sum(), cout_couple.loc[top_cher, "Après"].sum()
ax.set_title(f"Coût des 10 couples les plus chers de la baseline (camions réservés compris)\n"
             f"{eur(avant_top)} → {eur(apres_top)} ({pourcent(apres_top / avant_top - 1)})")
poignees, _ = ax.get_legend_handles_labels()
ax.legend(poignees + [Patch(facecolor="white", edgecolor="black", hatch="///")],
          ["Avant : baseline", "Après : planning consolidé", "dont camions réservés (poids inconnu)"], loc="lower right")
ax.grid(axis="x", alpha=0.5)
ax.set_axisbelow(True)
ax.margins(x=0.12)
plt.tight_layout()
plt.show()

#  Chiffres pour l'analyse 
detail_top = pd.DataFrame({"avant": cout_couple.loc[top_cher, "Avant"], "après": cout_couple.loc[top_cher, "Après"],
                           "dont réservés": cout_reserve.loc[top_cher, "Avant"]})
detail_top["économie"] = detail_top["avant"] - detail_top["après"]
detail_top["économie (%)"] = (detail_top["économie"] / detail_top["avant"] * 100).round(1)
print(detail_top[::-1].round(0))
reserve_top = detail_top["dont réservés"].sum()
print(f"Hors camions réservés : {eur(avant_top - reserve_top)} → {eur(apres_top - reserve_top)} "
      f"({pourcent((apres_top - reserve_top) / (avant_top - reserve_top) - 1)})")
print(f"Part des 10 couples dans le coût de la baseline : {pourcent(avant_top / cout_couple['Avant'].sum())} ; "
      f"part dans l'économie totale : {pourcent((avant_top - apres_top) / (cout_couple['Avant'].sum() - cout_couple['Après'].sum()))}")

#### Analyse : les 10 couples les plus chers

**Les couples aux données complètes baissent fortement.** SUP_00085 → PLT_004, le couple le plus cher aux données complètes, passe de 7 429 € à 1 988 € (−73 %). SUP_00146 → PLT_002 (−68 %), SUP_00144 → PLT_001 (−57 %), SUP_00059 → PLT_003 (−57 %) et SUP_00118 → PLT_001 (−54 %) suivent la même tendance. Ce sont tous des couples qui livraient 3 à 5 jours par semaine dans la baseline : les regrouper sur un jour fixe est ce qui rapporte le plus.

**Les couples aux poids manquants ne baissent presque pas.** Pour SUP_00153 → PLT_004 et SUP_00001 → PLT_004, l'économie est nulle : leur coût vient entièrement, ou presque, des camions réservés (lignes sans poids), qu'on ne peut pas consolider. SUP_00129 ne gagne que 15 % vers PLT_003 et 3 % vers PLT_001, pour la même raison. Nous l'avions dit au milestone 2 : pour ces couples, l'économie passe d'abord par la qualité des données.

**Au total, ces 10 couples passent de 48 749 € à 31 415 € (−35,6 %).** Hors camions réservés, la baisse atteint **−59,1 %** (de 29 343 € à 12 010 €), bien plus que la moyenne. Ces 10 couples pèsent 27,5 % du coût de la baseline et apportent 28,3 % de l'économie totale.

#### Zoom sur une semaine : l'exemple du cours

On reprend l'exemple du document du cours (semaine du 16/11, 5 couples très actifs), cette fois avec notre planning : pour chaque couple, le coût actuel (baseline) et le coût après notre optimisation. Les camions réservés sont exclus, car ils sont identiques avant et après.

In [ ]:
# coût de 5 couples sur la semaine du 16/11, avant / après 
SEMAINE = 47   # semaine du lundi 16/11
COUPLES_DIAPO = ["SUP_00085 × PLT_004", "SUP_00144 × PLT_001", "SUP_00059 × PLT_003",
                 "SUP_00080 × PLT_004", "SUP_00149 × PLT_004"]          # les 5 couples de l'exemple du cours

# Camions de la semaine, hors camions réservés (identiques avant et après)
sem = camions[~camions["reserve"] & (camions["semaine"] == SEMAINE) & camions["couple"].isin(COUPLES_DIAPO)]
diapo = sem.pivot_table(index="couple", columns="version", values="cout", aggfunc="sum").reindex(COUPLES_DIAPO)
nb_camions = sem.pivot_table(index="couple", columns="version", values="camion_id", aggfunc="size").reindex(COUPLES_DIAPO)

total_av, total_ap = diapo["Avant"].sum(), diapo["Après"].sum()
c85 = diapo.loc["SUP_00085 × PLT_004"]

fig = plt.figure(figsize=(15, 6))
grille = fig.add_gridspec(3, 2, width_ratios=[1.9, 1], hspace=0.35, wspace=0.08)
ax = fig.add_subplot(grille[:, 0])

y = np.arange(len(COUPLES_DIAPO))[::-1]          # le premier couple en haut
h = 0.38
for decalage, version, nom in [(h / 2, "Avant", "Coût actuel (€ / semaine)"), (-h / 2, "Après", "Coût après consolidation")]:
    barres = ax.barh(y + decalage, diapo[version], height=h, color=COULEURS[version], label=nom)
    ax.bar_label(barres, labels=[nombre(v, 0) for v in diapo[version]], padding=4, fontsize=11)
ax.set_yticks(y, COUPLES_DIAPO, fontsize=11)
ax.set_title("Coût de transport, semaine du 16/11 (€)", fontsize=15, fontweight="bold")
ax.xaxis.set_major_formatter(lambda v, _: nombre(v, 0))
ax.grid(axis="x", alpha=0.4)
ax.set_axisbelow(True)
ax.spines[["top", "right"]].set_visible(False)
ax.margins(x=0.1)
ax.legend(loc="lower center", bbox_to_anchor=(0.5, -0.2), ncol=2, frameon=False, fontsize=10)

#  Les 3 cartes chiffrées 
cartes = [
    (f"{pourcent(total_ap / total_av - 1, 0)}",
     f"coût des 5 couples : {eur(total_av)} → {eur(total_ap)} / semaine", "#e0a84f"),
    (eur(total_av - total_ap), "économisés sur une semaine pour 5 couples", "#e8eff1"),
    (eur(c85["Avant"] - c85["Après"]),
     f"SUP_00085 × PLT_004 : {int(nb_camions.loc['SUP_00085 × PLT_004', 'Avant'])} camions ({nombre(c85['Avant'], 0)} €)\n→ "
     f"{int(nb_camions.loc['SUP_00085 × PLT_004', 'Après'])} camion le lundi ({nombre(c85['Après'], 0)} €)", "#e8eff1"),
]
for i, (chiffre, texte, couleur) in enumerate(cartes):
    carte = fig.add_subplot(grille[i, 1])
    carte.set_facecolor(couleur)
    carte.set_xticks([]), carte.set_yticks([])
    for bord in carte.spines.values():
        bord.set_visible(False)
    carte.text(0.06, 0.62, chiffre, fontsize=26, fontweight="bold", color="#1f3b4d", transform=carte.transAxes)
    carte.text(0.06, 0.15, texte, fontsize=10.5, color="#1f3b4d", transform=carte.transAxes, wrap=True)

fig.text(0.66, 0.02, "Camions réservés exclus (identiques avant et après).", fontsize=9, color="dimgray")
plt.show()

#  Chiffres pour l'analyse 
diapo["camions avant"] = nb_camions["Avant"]
diapo["camions après"] = nb_camions["Après"]
print(diapo.round(2).to_string())
print(f"Total : {eur(total_av)} → {eur(total_ap)} ({pourcent(total_ap / total_av - 1)}), économie {eur(total_av - total_ap)}")
print("Jour du camion après :", sem[sem["version"] == "Après"].groupby("couple")["date"].first().dt.strftime("%d/%m").to_dict())

#### Analyse : la semaine du 16/11

**Chaque couple passe de 5 camions à 1 seul dans la semaine.** Ces 5 couples livraient tous les jours dans la baseline. Avec notre planning, chacun livre une seule fois, le jour fixe de son profil. Le coût des 5 couples passe de **3 994 € à 1 263 € pour la semaine (−68 %)**, soit **2 732 € économisés**. Pour SUP_00085 → PLT_004, les 5 camions (1 215 €) sont remplacés par un seul le lundi 16/11 (331 €) : 884 € d'économie sur une seule semaine.

**On retrouve exactement les chiffres de l'exemple du cours, bien que notre moteur n'ait pas fait les mêmes choix.** Pour 4 couples sur 5, il ne livre pas le même jour que le cours, et le camion ne transporte donc pas exactement le même poids. Mais tous ces camions dépassent 4 000 kg : dans cette dernière tranche de la grille, le prix ne dépend plus que de la distance. Un seul camion par semaine coûte donc le même prix, quel que soit son jour. Comme dans le cours, les camions réservés sont exclus : ils sont identiques avant et après.

**Ces 5 couples sont les cas les plus favorables** : ils livrent beaucoup, tous les jours, et ont des données complètes. Leur économie (−68 %) est donc bien supérieure à celle de l'ensemble du planning (−34,5 %).

#### Baseline, estimation et planning final, par semaine et par usine

Au milestone 2, on avait estimé le gain possible avec un algorithme de base : chaque camion part le jour du plus ancien besoin et regroupe les 7 jours suivants, sans jours fixes ni lissage. C'est le **potentiel** de la consolidation. On recalcule ici cette estimation avec le même algorithme, puis on compare les trois situations : la baseline (avant), l'estimation (le potentiel) et le planning final du moteur (le réalisé). Les camions réservés sont comptés dans les trois : ils sont identiques.

In [ ]:
# Recalcul de l'estimation du milstone 2 
FENETRE = pd.Timedelta(days=7)   # avance maximale d'une livraison sur son besoin (SF-06)


def consolider_couple_estimation(lignes):
    """Camions d'un couple. Les lignes sont triées par date de besoin, puis du plus lourd au plus léger.
    Un camion part le jour du plus ancien besoin restant et prend les besoins des 7 jours suivants,
    tant qu'il reste de la place."""
    restantes = list(lignes.itertuples())
    camions_couple = []
    while restantes:
        date = restantes[0].need_date
        poids, volume, contenu, i = 0.0, 0.0, [], 0
        while i < len(restantes):
            ligne = restantes[i]
            if ligne.need_date > date + FENETRE:
                break                                  # hors fenêtre : les suivantes aussi, camion fermé
            if poids + ligne.weight > POIDS_MAX or volume + ligne.volume > VOLUME_MAX:
                i += 1                                 # ne rentre pas : on essaie la ligne suivante
                continue
            poids, volume = poids + ligne.weight, volume + ligne.volume
            contenu.append(ligne.Index)
            restantes.pop(i)
            i = 0                                      # on repart de la plus ancienne ligne restante
        camions_couple.append({"date": date, "poids": poids, "volume": volume, "lignes": contenu})
    return camions_couple


# Lignes au poids connu (silver + quarantaine) ; la ligne de plus de 24,5 t reste exclue, comme au milestone 2
a_consolider = (besoins[besoins["weight"].notna() & (besoins["weight"] <= POIDS_MAX)]
                .sort_values(["plant", "supplier_id", "need_date", "weight"], ascending=[True, True, True, False]))
estimation = pd.DataFrame([{"supplier_id": supplier, "plant": plant, **camion}
                           for (plant, supplier), lignes_couple in a_consolider.groupby(["plant", "supplier_id"])
                           for camion in consolider_couple_estimation(lignes_couple)])
estimation["reserve"] = False

# Contrôles (les mêmes qu'au milestone 2)
lignes_livrees = estimation["lignes"].explode()
assert (estimation["poids"] <= POIDS_MAX).all() and (estimation["volume"] <= VOLUME_MAX).all()   # capacité
assert lignes_livrees.is_unique and set(lignes_livrees) == set(a_consolider.index)              # chaque ligne une fois
assert abs(estimation["poids"].sum() - a_consolider["weight"].sum()) < 1e-6                     # aucun poids perdu

# Les lignes sans poids gardent leur camion réservé, le jour du besoin, comme dans la baseline
estimation = pd.concat([estimation.drop(columns="lignes"), avant[avant["reserve"]].drop(columns="camion_id")],
                       ignore_index=True)
estimation["camion_id"] = range(1, len(estimation) + 1)

# Les trois situations, avec les mêmes indicateurs (fonction preparer de la préparation)
NOMS = {"Avant": "Baseline", "Estimation": "Estimation (milestone 2)", "Après": "Planning final"}
trois = pd.concat([camions[camions["version"] == "Avant"], preparer(estimation, "Estimation"),
                   camions[camions["version"] == "Après"]], ignore_index=True)
trois["version"] = trois["version"].map(NOMS)
COULEURS_3 = {"Baseline": COULEURS["Avant"], "Estimation (milestone 2)": "#e0a84f", "Planning final": COULEURS["Après"]}

trois.groupby("version", sort=False).agg(camions=("camion_id", "size"), dont_reserves=("reserve", "sum"),
                                         cout=("cout", "sum")).style.format({"cout": eur})

In [ ]:
#  Camions et coût par semaine 
camions_sem = trois.pivot_table(index="semaine", columns="version", values="camion_id", aggfunc="size", fill_value=0)[list(NOMS.values())]
cout_sem = trois.pivot_table(index="semaine", columns="version", values="cout", aggfunc="sum", fill_value=0)[list(NOMS.values())]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for ax, table, titre, fmt in [(axes[0], camions_sem, "Camions par semaine", lambda v: nombre(v, 0)),
                              (axes[1], cout_sem, "Coût de transport par semaine (€)", lambda v: nombre(v / 1000, 0) + " k")]:
    table.plot(kind="bar", ax=ax, color=[COULEURS_3[n] for n in table.columns], edgecolor="black", rot=0, width=0.85)
    for conteneur in ax.containers:
        ax.bar_label(conteneur, labels=[fmt(v) if v else "" for v in conteneur.datavalues], fontsize=7, padding=2)
    ax.set_xticks(range(len(table)), [f"S{s}" for s in table.index])
    ax.set_title(titre)
    ax.set_xlabel("Semaine")
    ax.yaxis.set_major_formatter(lambda v, _: nombre(v, 0))
    ax.margins(y=0.1)
    ax.grid(axis="y", alpha=0.4)
    ax.set_axisbelow(True)
    ax.legend(title=None)
fig.suptitle("Baseline, estimation du milestone 2 et planning final, par semaine (camions réservés compris)", fontsize=14)
plt.tight_layout()
plt.show()

#  Tableau par usine 
par_usine = trois.pivot_table(index="plant", columns="version", values="cout", aggfunc="sum")[list(NOMS.values())]
nb_usine = trois.pivot_table(index="plant", columns="version", values="camion_id", aggfunc="size")[list(NOMS.values())]
par_usine.loc["Total"] = par_usine.sum()
nb_usine.loc["Total"] = nb_usine.sum()
base, est, fin = (par_usine[n] for n in NOMS.values())
tableau_usines = pd.DataFrame({
    "Camions (baseline -> estimation -> final)": [f"{a} -> {b} -> {c}" for a, b, c in nb_usine.values],
    "Coût baseline": base, "Coût estimation": est, "Coût final": fin,
    "Gain estimé (potentiel)": (base - est) / base,
    "Gain réalisé": (base - fin) / base,
    "Part du potentiel captée": (base - fin) / (base - est),
    "Part du gain réalisé": (base - fin) / (base - fin)["Total"],
})
tableau_usines.style.format({"Coût baseline": eur, "Coût estimation": eur, "Coût final": eur,
                             "Gain estimé (potentiel)": pourcent, "Gain réalisé": pourcent,
                             "Part du potentiel captée": pourcent, "Part du gain réalisé": pourcent})

#### Analyse : baseline, estimation et planning final

**Le planning final suit de près l'estimation du milestone 2.** Il compte 787 camions et coûte 116 118 €, contre 692 camions et 109 513 € pour l'estimation, et 1 741 camions et 177 321 € dans la baseline. Il réalise un gain de **61 203 € (−34,5 %)**, soit **90 % du gain estimé** (67 807 €, −38,2 %). Les 10 % restants (6 604 €) sont le prix des règles que l'estimation ignorait : des jours fixes pour chaque couple, le lissage de la charge des usines, et l'obligation de livrer chaque jour du profil au moins une fois.

**Semaine par semaine, le planning final reste proche de l'estimation.** Il envoie entre 104 et 164 camions par semaine (S46 à S50), contre 239 à 402 dans la baseline. Le pic de S48 disparaît : 402 camions et 39 899 € dans la baseline, 164 camions et 24 200 € dans le planning final, contre 134 camions et 21 800 € dans l'estimation.

**Certaines semaines coûtent moins cher que l'estimation** (S46 et S51). Ce n'est pas une erreur : l'estimation n'est un maximum que sur l'ensemble de la période, car les camions ne partent pas les mêmes jours. Les besoins du début de S46 sont livrés en avance dès S45 (57 camions du 03 au 06/11, une semaine qui n'existait pas dans la baseline), et ceux de S51 en partie en S50.

**Par usine, le moteur capte entre 87 % et 92 % du gain estimé.** PLT_004 reste la cible prioritaire annoncée au milestone 2 : c'est elle qui gagne le plus (−39,8 %, 38,1 % du gain total), et elle capte 92 % de son potentiel. PLT_002 en capte le moins (87 %, 237 camions au lieu des 197 de l'estimation) : c'est l'usine dont la charge était la plus irrégulière (pic de S48), donc celle où les jours fixes et le lissage imposent le plus de camions en plus. PLT_003 gagne toujours le moins en pourcentage (−27,0 %), à cause de ses camions réservés, que ni l'estimation ni le moteur ne peuvent consolider.

#### Réalisé vs potentiel : d'où vient l'écart

Le graphe précédent montre que le planning final reste au-dessus de l'estimation du milestone 2. Ce tableau décompose cet écart, étape par étape du moteur : on part du potentiel (consolidation sans contrainte de jours), puis on ajoute une à une les règles métier. Les coûts viennent du bilan du milestone 3 (`data/gold/bilan.csv`) et comprennent les camions réservés.

In [ ]:
#  Réalisé vs potentiel, étape par étape 
BILAN_PATH = "../data/gold/bilan.csv"   # bilan des versions (milestone 3)
bilan = pd.read_csv(BILAN_PATH).set_index("Version")

baseline = bilan.loc["Baseline", "Coût"]
potentiel = bilan.loc["Algorithme de base (sans jours fixes)", "Coût"]      # = estimation du milestone 2
jours_fixes = bilan.loc["1 ou 2 jours fixes", "Coût"]
lissage = bilan.loc["1 ou 2 jours fixes + lissage", "Coût"]
realise = bilan.loc["1 ou 2 jours fixes + lissage + report", "Coût"]       # planning final
cout_reserves = camions.loc[camions["reserve"] & (camions["version"] == "Avant"), "cout"].sum()   # identique partout

etapes = pd.DataFrame([
    ("Baseline (situation actuelle)", baseline, "—"),
    ("Potentiel : consolidation sans contrainte de jours", potentiel, eur(potentiel - baseline)),
    ("+ jours fixes (1 ou 2 par couple, chaque jour du profil utilisé)", jours_fixes, f"+{eur(jours_fixes - potentiel)}"),
    ("+ lissage de la charge des usines", lissage, f"+{eur(lissage - jours_fixes)}"),
    ("+ report des lignes (réalisé, planning final)", realise, eur(realise - lissage)),
], columns=["Étape", "Coût", "Écart avec l'étape précédente"])
etapes["Économie vs baseline"] = [eur(baseline - c) + f" ({pourcent((baseline - c) / baseline)})" if c != baseline else "—"
                                  for c in etapes["Coût"]]
etapes["Coût"] = etapes["Coût"].map(eur)
display(etapes.style.hide(axis="index"))

print(f"Part du potentiel captée : {eur(baseline - realise)} sur {eur(baseline - potentiel)} = "
      f"{pourcent((baseline - realise) / (baseline - potentiel))}")
print(f"Coût des contraintes métier (jours fixes + lissage + report) : {eur(realise - potentiel)}")
print(f"Camions réservés (poids inconnu), identiques dans toutes les versions : {eur(cout_reserves)}, "
      f"soit {pourcent(cout_reserves / baseline)} de la baseline et {pourcent(cout_reserves / realise)} du planning final")

#### Analyse : réalisé vs potentiel

**Le planning final capte 90 % du gain possible.** Sans aucune contrainte de jours, la consolidation ferait économiser 67 807 € (−38,2 %) : c'est l'estimation du milestone 2. Le planning final économise **61 203 € (−34,5 %)**. Les règles métier coûtent donc **6 604 € sur les 6 semaines**, soit 10 % du gain possible.

**Les jours fixes représentent presque tout ce coût (+5 573 €).** Dans l'estimation, chaque camion part le jour exact du premier besoin. Avec un jour fixe, il part le dernier jour fixe avant ce besoin : sa fenêtre de 7 jours couvre donc moins de besoins à venir, et il faut plus de camions (766 au lieu de 692). La règle qui oblige à livrer au moins une fois chaque jour du profil rend un deuxième jour fixe peu intéressant : il impose des livraisons supplémentaires. Le moteur donne donc un seul jour fixe à 172 couples sur 173.

**Le lissage coûte peu au regard de ce qu'il apporte (+1 246 €).** Il ajoute 21 camions, mais ramène les semaines hors tolérance de 20 sur 20 à seulement 2. Sans lui, les couples d'une même usine choisiraient souvent le même jour, et les quais seraient saturés certains jours et vides d'autres. Le report des lignes sur le jour fixe suivant fait regagner 215 €, sans ajouter de camion.

**Le principal potentiel restant n'est pas dans l'algorithme, mais dans les données.** Les 134 lignes sans poids coûtent 33 506 € dans toutes les versions, soit 28,9 % du planning final, et le moteur ne peut pas les consolider. C'est 5 fois plus que le coût de toutes les règles métier réunies : renseigner les poids manquants rapporterait bien plus que n'importe quelle amélioration du moteur.

---

## Conclusion

**La consolidation change profondément la façon de livrer.**
- **Moins de camions, plus lourds** : on passe de 1 741 à 787 camions (÷ 2,2). Le poids médian d'un camion passe de 323 kg à 687 kg, et la part des camions au-delà de 4 000 kg (la dernière tranche de la grille, où le prix est fixe) de 4,4 % à 15,2 %. Ces camions transportent désormais 69 % du poids.
- **Moins cher à la tonne** : le coût moyen d'une tonne transportée passe de 91 € à 52 €. Les camions de moins de 500 kg, qui coûtaient 39 % du budget pour 13 % du poids, n'en coûtent plus que 20 %.
- **Des livraisons régulières** : 172 couples sur 173 livrent un seul jour fixe par semaine. Les couples les plus actifs passent de 5 livraisons par semaine à une seule. En moyenne, un couple livre 1,1 jour par semaine au lieu de 2,0.
- **Des usines moins chargées** : les pics de réception disparaissent. Le jour le plus chargé de PLT_002 passe de 54 à 15 camions, et celui de toutes les usines réunies de 118 à 35.

**L'économie est importante et proche du maximum possible.**
- Le coût total passe de **177 321 € à 116 118 € sur les 6 semaines, soit 61 203 € économisés (−34,5 %)**.
- C'est **90 % du potentiel** estimé au milestone 2 (67 807 €, sans aucune contrainte de jours). Les règles métier coûtent 6 604 € : surtout les jours fixes (+5 573 €), puis le lissage (+1 246 €), qui ramène les semaines hors tolérance de 20 à 2 sur 20.
- L'exemple du cours est retrouvé sur nos données : sur la semaine du 16/11, les 5 couples de l'exemple passent de 3 994 € à 1 263 € (−68 %).
- Toutes les usines y contribuent. PLT_004 apporte la plus grosse part (23 345 €, 38 % de l'économie).

**Le principal gisement restant est la qualité des données.** Les 134 lignes sans poids partent chacune dans un camion réservé, qui ne peut pas être consolidé : elles coûtent 33 506 € dans toutes les versions, soit 29 % du planning final, cinq fois plus que toutes les règles métier réunies. Ce sont elles qui expliquent les couples et les usines qui économisent le moins (SUP_00129, SUP_00153, PLT_003). Hors camions réservés, chaque usine économise entre 38 % et 46 %. Renseigner ces poids est le levier le plus efficace pour aller plus loin, devant toute amélioration de l'algorithme.

**Limites.** Le remplissage reste faible dans l'absolu (2,8 % en médiane), car la plupart des couples ont moins de 4 t de besoins par semaine. Pour remplir réellement les camions, il faudrait sortir du périmètre du projet : mélanger plusieurs fournisseurs dans un même camion (tournées), ou livrer plus de 7 jours en avance. Enfin, la ligne de plus de 24,5 t (SUP_00074, 26,9 t) reste exclue en attendant la réponse du client.